# OCR a scanned PDF with `pdf_inspector`

`process_pdf` only reads a PDF's embedded text layer, so on a scanned document it
returns `pdf_type == "scanned"` and `markdown is None`. To get text you need
`process_pdf_with_ocr`, which rasterizes each page and runs a recognition model
over the pixels.

That needs two native libraries which `pdf-inspector` does **not** install for you:

| Library | Provided by | Env var |
|---|---|---|
| `libpdfium.dylib` | `pip install pypdfium2` | `PDFIUM_LIB_PATH` |
| `libonnxruntime.dylib` | `pip install onnxruntime` | `ORT_DYLIB_PATH` |

> **Known limitation:** the bundled `pp-ocrv6_small_rec` model's character
> dictionary has no Vietnamese stacked tone marks (`ớ ợ ầ ề ế ộ ừ ữ ổ ị ả`), so
> Vietnamese comes out stripped — `Lớp thượng bì` becomes `Lóp thưng bì`. The
> reported confidence stays high (~0.96) because the model is confident about the
> characters it *can* emit. See the last cell for the workaround.


In [2]:
import os
import pathlib

# pdf_inspector dlopen()s these at the first OCR call, so point at them *before*
# calling process_pdf_with_ocr. setdefault means a path already set in the kernel
# spec (~/Library/Jupyter/kernels/mfin/kernel.json) wins over these.
import onnxruntime
import pypdfium2_raw

os.environ.setdefault(
    "PDFIUM_LIB_PATH",
    str(pathlib.Path(pypdfium2_raw.__file__).parent / "libpdfium.dylib"),
)
os.environ.setdefault(
    "ORT_DYLIB_PATH",
    str(next(pathlib.Path(onnxruntime.__file__).parent.joinpath("capi").glob("libonnxruntime*.dylib"))),
)

import pdf_inspector

# The kernel's working directory is NOT the notebook's folder (VS Code usually
# starts it in your home dir), so pin an absolute path rather than a relative one.
CONTENT_DIR = pathlib.Path(
    "/Users/ghn-lap-00084/Documents/ISB/MFIN/AI-For-Beginners/content"
)
PDF = CONTENT_DIR / "Giải phẫu 2022 (bản mới).pdf"

if not PDF.exists():
    raise FileNotFoundError(f"{PDF}\n(kernel cwd is {pathlib.Path.cwd()})")

PAGES = [20, 21]   # 1-indexed pages to sample; used by the OCR cells below

print("pdf        :", PDF.name)
print("size       :", f"{PDF.stat().st_size / 1024**2:.0f} MB")
print("cwd        :", pathlib.Path.cwd())
print("pdfium     :", os.environ["PDFIUM_LIB_PATH"])
print("onnxruntime:", os.environ["ORT_DYLIB_PATH"])


pdf        : Giải phẫu 2022 (bản mới).pdf
size       : 421 MB
cwd        : /Users/ghn-lap-00084
pdfium     : /Users/ghn-lap-00084/Documents/ISB/MFIN/.venv/lib/python3.14/site-packages/pypdfium2_raw/libpdfium.dylib
onnxruntime: /Users/ghn-lap-00084/Documents/ISB/MFIN/.venv/lib/python3.14/site-packages/onnxruntime/capi/libonnxruntime.1.30.0.dylib


In [3]:
# Step 1 — classify. No OCR yet, so this is fast even on a 733-page file.
result = pdf_inspector.process_pdf(str(PDF))

print("pdf_type         :", result.pdf_type)   # text_based | scanned | image_based | mixed
print("page_count       :", result.page_count)
print("confidence       :", result.confidence)
print("has_markdown     :", result.markdown is not None)
print("pages_needing_ocr:", len(result.pages_needing_ocr), "of", result.page_count)
print("complex_layout   :", result.is_complex_layout)
print("pages_with_tables:", result.pages_with_tables[:10], "...")

# markdown is None for a scanned PDF — that is the whole reason we need OCR below.
print("\nmarkdown:", repr(result.markdown))


pdf_type         : scanned
page_count       : 733
confidence       : 0.949999988079071
has_markdown     : False
pages_needing_ocr: 733 of 733
complex_layout   : False
pages_with_tables: [] ...

markdown: None


In [4]:
# Step 2 — OCR. PAGES comes from the setup cell; change it there. The full 733
# pages take several minutes, and model weights download on the very first call.
ocr = pdf_inspector.process_pdf_with_ocr(str(PDF), page_numbers=PAGES, dpi=150.0)

print("routed to ocr :", ocr.pages_routed_to_ocr)
print("render_time_ms:", ocr.render_time_ms)
print("ocr_time_ms   :", ocr.ocr_time_ms)
print("chars produced:", len(ocr.markdown))


routed to ocr : [20, 21]
render_time_ms: 159
ocr_time_ms   : 2395
chars produced: 4433


In [5]:
# Step 3 — SHOW THE TEXT. Unlike PdfResult.markdown, OcrPdfResult.markdown is a
# plain str and is never None, so this is where the recognized text lives.
from IPython.display import Markdown, display

display(Markdown(ocr.markdown))


Lóp thưng bì: Các t bào Tàng sirngNông sirng đă cht 个 -Tàng sirng Tàng sáng Tàng ht Các hąt lá —Tàng sáng

- T bào sng —Tàng hat
Tàng gai Tàng gai T bào -Langerhans -T bào Merkel —Tàng dáy Đǐa xúc giác -Thàn kinh cám giác

- T bào hc t
-Lóp bi Tàng dáy-

- Lóp bì
Sâu LM240x

(a) Các tàng t bào chính ca thương bì (b) Ånh phóng đąi duói kính hin vi
Hinh 1. Các loi t bào và các tng ca thưng bì [16] Khi t bào hc t không tông hp đưc men tyrosinase đ sån xut melanin, không có melanin mt, tóc và da gây nên bch tng (albinism). Mt mt phàn hay toàn b t bào hc t các mng da to nên các mng da trng không đu goi là bch bin (vitiligo).

2. Bì (dermis) Lóp bì đưc cu to bng mô liên kt dày đc không đu cha các si collagen và si trun. Mng lưi si này to nên s dai bn, có th kéo giãn ra và thu hi d dàng. Bì dày hơn thưng bì nhiu và đ dày này bin đi tùy tng vùng, dày nht gan tay và gan chân. Thành phn t bào trong lóp bì ch yu là các nguyên bào si cùng vói mt s đi thc bào và mt ít t bào m nm gn ranh giói vi lóp m dưi da. Các thành phàn vùi trong lóp bì bao gồm các mch máu, các thàn kinh, các tuyn và các nang lông. Lóp bì là lóp thit yu cho s sng còn ca lóp thưng bì và gia hai lóp này có nhiu mi liên h v cu trúc và chc năng quan trong. Da vào cu trúc mô, có th chia bì thành mt lp nhú nông và mt lóp lưi sâu. Lóp nhú (papillary layer) to nên khong mt phàn năm chiu dày ca bì. Lóp này cha nhng si collagen mnh và các si trun mn. Din tích b mt ca lóp này tăng lên nhò có các nhú (papillae), là nhng cu trúc nh hinh núm vú nhô vào mt dưi ca thưng bi. Tt cà các nhú cha các quai mao mch. Mt s cũng cha các th th xúc giác goi là các tiu th Meissner. Có nhú bì cha các đu tn cùng t do, các nhánh cành mà thiu s bit hóa cu trúc bên ngoài.

Lóp lưói (reticular layer) bao gồm nhng bó si collagen dày, các nguyên bào si ri rác, và nhng t bào không c đnh (như đi thc bào). Có th thy mt s t bào m có mt phn sâu nht cůa lóp này, cùng vói mt s si chun to. Các si collagen trong lóp lưi đưc sp xp theo kiu mng lưi và có s sp xp đu hon so vi lóp nhú, vì th mà có tên là lóp lưói. Hưóng đu hon cůa các si collagen giúp da kháng li s kéo giãn tt hon. Các mch máu, các thn kinh, các nang lông, các tuyn bã và các tuyn mò hôi chim nhng khoang gia các si. S kt hp cůa các si collagen và các si chun làm cho da chc, có kh năng giãn ra và tr li hình dng ban đu. Trên b mt ca gan tay, ngón tay, gan chân có nhiu gò và rãnh. Chúng hin ra như nhng đưòng thng hay như mt mãu ca các đưòng cong, vòng xon, như đu các ngón tay. Nhng gò thưng bì này đưc to ra trong tháng th ba ca s phát trin thai do s lồi xuông dưói cůa thưng bì vào lóp bì gia các nhú bì ca lóp nhú. Các gò thưng bì to ra s liên kt chc gia thưng bì và bì trong mt vùng cn sc bn cơ hc. Các gò thưng bì cng làm tăng din tích b mt ca thưng bì và, như vy, làm tăng kh năng nm cht bng cách làm tăng lc ma sát. Cui cùng, gò thưng bì làm tăng din tích b mt, qua đó làm tăng s lưng các tiu th Meissner và làm tăng xúc giác. Mu gò thưng bì ngón tay in trên mt b mt nhn là biu hin ca đc đim di truyn đon nht cho mi cá th, không thay đi trong đi sng và là co s cho nhn dng. Ngoài to nên nhng gò thưng bì, b mt nhú ca bì còn làm tǎng tip xúc bè mt gia bì và thưng bì, tăng s nuôi dưõng cůa bì cho thưng bì bên trên. S lồng cài vào nhau gia nhng nhú bì và gò thưng bì to nên liên kt khóp cc kì chc, chng li lc xé tách hai lóp vi nhau. nhng vùng nht đnh ca co th, các si collagen trong lóp lưi ca bì có xu hưóng sp xp theo hưóng này nhiu hon theo hưóng khác. Hưóng sp xp nhiu hon này goi là hưóng tri, thưòng do sc căng t nhiên tác đng lên vùng da đó, ví d như s co co bên dưi, quy đnh. Hưóng tri thưng vuông góc vi hưóng ca các si co bên dưi, và đưc goi là các đưòng căng (tension lines) hay đưòng xe (cleavage lines). Trong phu thut thm mĩ, đưòng rch da dc theo đưng căng, tc theo hưóng tri ca các si collagen thì mau lin và so nh. Trái li, nu đưòng rch ct ngang qua các si collagen, sęo s to và xu. Các vt rn da (stretch marks). Khi da b kéo giãn ra quá mc, như da bng trong thòi kì mang thai, có th gây nhng tồn thưong bên trong. Liên kt bên gia các si collagen b đt và nhng mch máu nh ca da b võ. Đó là lý do vt rn lúc đu trông như các vt hoi đ. V sau, sęo đưc hinh thành nhng chõ tồn thưong, các vt rn xut hin như nhng vt trng có ánh bc.

3. CÁC CÁU TRÚC PH CÚA DA Các cu trúc ph cůa da, bao gòm lông, các tuyn da và móng, phát trin tù thưng bì ca phôi. Chúng góp phn vào các chc năng bo v và điu nhit ca da.


In [6]:
# Step 4 — per-page text plus provenance, so you can tell *which* pages the OCR
# struggled on. provenance.source is "native" (embedded text was fine), "ocr"
# (recognized from pixels) or "fused" (both merged).
for page in ocr.pages:
    prov = page.provenance
    conf = "n/a" if prov.ocr_confidence is None else f"{prov.ocr_confidence:.3f}"
    model = prov.ocr_model.name if prov.ocr_model else "-"

    print("=" * 78)
    print(f"page {page.page_number} | source={prov.source} | confidence={conf}")
    print(f"model={model}  dpi={prov.render_dpi}  ocr_ms={prov.timings.ocr_ms}")
    if prov.warnings:
        print("warnings:", prov.warnings)
    print("-" * 78)
    print(page.markdown[:1200])


page 20 | source=ocr | confidence=0.956
model=pp-ocrv6-small  dpi=150.0  ocr_ms=2093
------------------------------------------------------------------------------
Lóp thưng bì: Các t bào Tàng sirngNông sirng đă cht 个 -Tàng sirng Tàng sáng Tàng ht Các hąt lá —Tàng sáng

- T bào sng —Tàng hat
Tàng gai Tàng gai T bào -Langerhans -T bào Merkel —Tàng dáy Đǐa xúc giác -Thàn kinh cám giác

- T bào hc t
-Lóp bi Tàng dáy-

- Lóp bì
Sâu LM240x

(a) Các tàng t bào chính ca thương bì (b) Ånh phóng đąi duói kính hin vi
Hinh 1. Các loi t bào và các tng ca thưng bì [16] Khi t bào hc t không tông hp đưc men tyrosinase đ sån xut melanin, không có melanin mt, tóc và da gây nên bch tng (albinism). Mt mt phàn hay toàn b t bào hc t các mng da to nên các mng da trng không đu goi là bch bin (vitiligo).

2. Bì (dermis) Lóp bì đưc cu to bng mô liên kt dày đc không đu cha các si collagen và si trun. Mng lưi si này to nên s dai bn, có th kéo giãn ra và thu hi d dàng. Bì dày hơn thưng bì nhiu và đ dày này bin đi

In [7]:
# Step 5 — full document to a file. Guarded, because this is the slow one and you
# do not want 733 pages of text living inside the .ipynb.
RUN_FULL = False

if RUN_FULL:
    full = pdf_inspector.process_pdf_with_ocr(str(PDF), dpi=150.0)
    out = PDF.with_suffix(".md")
    out.write_text(full.markdown, encoding="utf-8")
    print(f"wrote {out}  ({len(full.markdown):,} chars)")
    print("pages routed to ocr:", len(full.pages_routed_to_ocr), "of", full.page_count)
    print("total time:", full.processing_time_ms / 1000, "s")
else:
    print("RUN_FULL is False — set it to True to OCR all pages.")


RUN_FULL is False — set it to True to OCR all pages.


In [8]:
# Step 6 — proof that the diacritic loss is a charset limit, not a resolution or
# dpi problem. Raising dpi cannot invent characters the model cannot emit.
dict_path = (
    pathlib.Path.home()
    / "Library/Caches/pdf-inspector/models/pp-ocrv6-small/oar-ocr-v0.7.0/ppocrv6_dict.txt"
)

if dict_path.exists():
    charset = set(dict_path.read_text(encoding="utf-8").split("\n"))
    print(f"dictionary entries: {len(charset):,}\n")
    for label, chars in [
        ("single diacritic ", "áàâăóôéèđ"),
        ("stacked (Vietnamese)", "ớợầềếộừữổịả"),
    ]:
        have = "".join(c for c in chars if c in charset) or "(none)"
        miss = "".join(c for c in chars if c not in charset) or "(none)"
        print(f"{label}  present: {have}")
        print(f"{label}  MISSING: {miss}")
else:
    print("dictionary not downloaded yet — run the OCR cell above first")


dictionary entries: 18,709

single diacritic   present: áàâăóôéèđ
single diacritic   MISSING: (none)
stacked (Vietnamese)  present: (none)
stacked (Vietnamese)  MISSING: ớợầềếộừữổịả


---

## Getting correct Vietnamese: macOS Vision

The PP-OCRv6 charset problem above is not fixable by tuning `dpi` or `mode`, and
it is **not** fixed by PaddleOCR either — `PaddleOCR(lang='vi')` in paddleocr 3.x
downloads `PP-OCRv6_medium_rec`, whose dictionary is the same 18,708 entries with
the same missing stacked tone marks. `lang=` only affects detection there.

macOS ships an on-device OCR engine (Vision.framework) that does support
Vietnamese as `vi-VT`, with no model download. Same sentence, three engines:

| Engine | Output |
|---|---|
| `pdf_inspector` (PP-OCRv6 small) | `Lóp bì đưc cu to bng mô liên kt dày đc` |
| PaddleOCR `lang='vi'` (PP-OCRv6 medium) | `Lóp bì đưc cu to bng mô liên kt dày đc` |
| **macOS Vision `vi-VT`** | **`Lớp bì được cấu tạo bằng mô liên kết dày đặc`** |

Requires `pip install pyobjc-framework-Vision pyobjc-framework-Quartz` (small,
and it works on Python 3.14 so it runs in this same kernel).


In [9]:
# Step 7 — Vietnamese OCR via macOS Vision. pypdfium2 rasterizes the page,
# Vision recognizes it. Nothing is downloaded and nothing leaves the machine.
import io

import pypdfium2 as pdfium
import Quartz
import Vision
from Foundation import NSData

VISION_LANGS = ["vi-VT"]   # Apple's code for Vietnamese
SCALE = 3                  # 72dpi * 3 = 216 dpi; 2 is faster, 4 is slower


def vision_ocr(img, languages=VISION_LANGS):
    """OCR a PIL image, returning [(text, confidence), ...] one entry per line."""
    png = io.BytesIO()
    img.save(png, format="PNG")
    raw = png.getvalue()

    data = NSData.dataWithBytes_length_(raw, len(raw))
    source = Quartz.CGImageSourceCreateWithData(data, None)
    cg_image = Quartz.CGImageSourceCreateImageAtIndex(source, 0, None)

    request = Vision.VNRecognizeTextRequest.alloc().init()
    request.setRecognitionLevel_(0)          # 0 = accurate, 1 = fast
    request.setRecognitionLanguages_(languages)
    request.setUsesLanguageCorrection_(True)

    handler = Vision.VNImageRequestHandler.alloc().initWithCGImage_options_(cg_image, None)
    ok, err = handler.performRequests_error_([request], None)
    if not ok:
        raise RuntimeError(f"Vision failed: {err}")

    out = []
    for observation in request.results():
        best = observation.topCandidates_(1)[0]
        out.append((best.string(), best.confidence()))
    return out


def vision_ocr_page(doc, page_number, scale=SCALE):
    """OCR one 1-indexed page of an open PdfDocument."""
    image = doc[page_number - 1].render(scale=scale).to_pil().convert("RGB")
    return vision_ocr(image)


doc = pdfium.PdfDocument(str(PDF))

for page_number in PAGES:
    lines = vision_ocr_page(doc, page_number)
    mean_conf = sum(c for _, c in lines) / max(len(lines), 1)

    print("=" * 78)
    print(f"page {page_number}  [{len(lines)} lines, mean confidence {mean_conf:.3f}]")
    print("-" * 78)
    print("\n".join(text for text, _ in lines))


page 20  [51 lines, mean confidence 1.000]
------------------------------------------------------------------------------
Tầng sừng
Tầng sáng
Tầng hạt
-Các tế bào
sừng đã chết
Nông
-Các hạt lá
- Tế bào sừng
Lớp thượng bì:
-Tầng sừng
-Tầng sáng
-Tầng hạt
-Tầng gai
-Tầng đáy
Tầng gai+
Tế bào
Langerhans
Tế bào Merkel
Đĩa xúc giác
Thần kinh
cảm giác
Tầng đáy
- Tế bào hắc tố
- Lớp bì
-Lớp bì
Sâu
240x
(a) Các tầng tế bào chính của thượng bì
(b) Ảnh phóng đại dưới kính hiến vi
Hình 1. Các loại tế bào và các tầng của thượng bì [16]
Khi tế bào hắc tố không tổng hợp được men tyrosinase để sản xuất melanin,
không có melanin ở mắt, tóc và da gây nên bạch tạng (albinism). Mất một phần hay
toàn bộ tế bào hắc tố ở các mảng da tạo nên các mảng da trắng không đều gọi là bạch
biên (vitiligo).
2. Bì (dermis)
Lớp bì được cấu tạo bằng mô liên kết dày đặc không đều chứa các sợi collagen và
sợi trun. Mạng lưới sợi này tạo nên sự dai bền, có thế kéo giãn ra và thu hồi dễ dàng. Bì
dày hơn thượng bì nhiều và độ

In [11]:
# Step 8 — full document to Markdown with Vision. Guarded like Step 5.
# Writes incrementally so a crash or interrupt does not lose completed pages.
RUN_FULL_VISION = True

if RUN_FULL_VISION:
    import time

    out_path = PDF.with_name(PDF.stem + ".vi.md")
    doc = pdfium.PdfDocument(str(PDF))
    started = time.time()

    with out_path.open("w", encoding="utf-8") as fh:
        for page_number in range(1, len(doc) + 1):
            lines = vision_ocr_page(doc, page_number)
            fh.write(f"\n\n<!-- page {page_number} -->\n\n")
            fh.write("\n".join(text for text, _ in lines))
            fh.flush()

            if page_number % 25 == 0 or page_number == len(doc):
                elapsed = time.time() - started
                rate = page_number / elapsed
                eta = (len(doc) - page_number) / rate
                print(
                    f"{page_number}/{len(doc)} pages"
                    f"  {rate:.2f} pages/s"
                    f"  eta {eta / 60:.1f} min",
                    flush=True,
                )

    print(f"\nwrote {out_path}  ({out_path.stat().st_size / 1024:,.0f} KB)")
else:
    print("RUN_FULL_VISION is False — set it to True to OCR all pages to .vi.md")


25/733 pages  1.07 pages/s  eta 11.0 min
50/733 pages  1.02 pages/s  eta 11.1 min
75/733 pages  1.02 pages/s  eta 10.7 min
100/733 pages  1.05 pages/s  eta 10.1 min
125/733 pages  1.06 pages/s  eta 9.5 min
150/733 pages  1.09 pages/s  eta 8.9 min
175/733 pages  1.10 pages/s  eta 8.5 min
200/733 pages  1.08 pages/s  eta 8.2 min
225/733 pages  1.07 pages/s  eta 7.9 min
250/733 pages  1.06 pages/s  eta 7.6 min
275/733 pages  1.04 pages/s  eta 7.3 min
300/733 pages  1.04 pages/s  eta 6.9 min
325/733 pages  1.03 pages/s  eta 6.6 min
350/733 pages  1.03 pages/s  eta 6.2 min
375/733 pages  1.03 pages/s  eta 5.8 min
400/733 pages  1.02 pages/s  eta 5.4 min
425/733 pages  1.03 pages/s  eta 5.0 min
450/733 pages  1.03 pages/s  eta 4.6 min
475/733 pages  1.02 pages/s  eta 4.2 min
500/733 pages  1.02 pages/s  eta 3.8 min
525/733 pages  1.02 pages/s  eta 3.4 min
550/733 pages  1.03 pages/s  eta 3.0 min
575/733 pages  1.03 pages/s  eta 2.6 min
600/733 pages  1.03 pages/s  eta 2.1 min
625/733 pages  